# Leakage-safe temporal robustness report

> Each matrix row measures degradation of a TabPFN system re-contextualized using that row's reference-year data, not degradation of the original pre-2007 context.

Artifact-only notebook. It validates, filters, summarizes, and plots frozen outputs. It performs no model, SAE, target, rule, CAV, gradient, recurrence, bootstrap, or change-point fitting. Anchor views use cosine `0.60` and overlap `0.70`; future-selected sensitivity artifacts remain labeled `selection_scope=post_hoc_future`.

In [ ]:
from __future__ import annotations
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ARTIFACT_ROOT = Path(os.environ.get('TEMPORAL_ARTIFACT_DIR', 'stats/temporal_robustness'))
HEADLINE_COSINE = 0.60
HEADLINE_OVERLAP = 0.70


## Artifact discovery and validation

In [ ]:
def read_json(path: Path):
    with path.open(encoding='utf-8') as handle:
        return json.load(handle)

def resolve_parent(root: Path) -> Path | None:
    direct = root / 'parent_manifest.json'
    if direct.is_file():
        return direct
    candidates = sorted(root.glob('*/parent_manifest.json'), key=lambda path: path.stat().st_mtime)
    return candidates[-1] if candidates else None

PARENT_PATH = resolve_parent(ARTIFACT_ROOT)
parent = read_json(PARENT_PATH) if PARENT_PATH else None
if parent is None:
    print(f'No completed temporal run under {ARTIFACT_ROOT.resolve()}')
else:
    required = {'schema_version', 'estimand', 'successful_experiments', 'skipped_references', 'failed_experiments'}
    missing = required - set(parent)
    if missing:
        raise ValueError(f'Parent manifest missing fields: {sorted(missing)}')
    print('Valid experiments:', len(parent['successful_experiments']))
    display(pd.DataFrame(parent['skipped_references']))
    display(pd.DataFrame(parent['failed_experiments']))


## Load normalized artifact tables

In [ ]:
def load_tables(parent_manifest):
    tables = {}
    manifests = []
    for entry in parent_manifest.get('successful_experiments', []):
        manifest_path = Path(entry['manifest'])
        if not manifest_path.is_file() and PARENT_PATH:
            manifest_path = PARENT_PATH.parent / f"reference_{entry['reference_year']}" / f"split_{entry['patient_split_seed']}" / 'manifest.json'
        manifest = read_json(manifest_path)
        manifests.append(manifest)
        for name in manifest.get('artifacts', {}):
            csv_path = manifest_path.parent / f'{name}.csv'
            if csv_path.is_file():
                frame = pd.read_csv(csv_path)
                frame['reference_year'] = frame.get('reference_year', manifest['reference_year'])
                frame['patient_split_seed'] = frame.get('patient_split_seed', manifest['patient_split_seed'])
                tables.setdefault(name, []).append(frame)
    for name, csv_name in parent_manifest.get('aggregate_artifacts', {}).items():
        csv_path = Path(csv_name)
        if not csv_path.is_file() and PARENT_PATH:
            csv_path = PARENT_PATH.parent / 'aggregate' / f'{name}.csv'
        if csv_path.is_file() and name not in tables:
            tables[name] = [pd.read_csv(csv_path)]
    return manifests, {name: pd.concat(frames, ignore_index=True) for name, frames in tables.items()}

manifests, tables = load_tables(parent) if parent else ([], {})
pd.DataFrame([{'table': name, 'rows': len(frame)} for name, frame in sorted(tables.items())])


## Cohort/class support and factor-family membership

In [ ]:
support = tables.get('performance', pd.DataFrame())
support_columns = [column for column in ['reference_year', 'test_year', 'temporal_distance', 'patient_split_seed', 'cohort_view', 'sample_count', 'patient_count', 'death_count', 'survivor_count', 'failure_reason'] if column in support]
display(support[support_columns] if support_columns else support)
families = tables.get('factor_families', tables.get('family_members', pd.DataFrame()))
display(families)
recurrence = tables.get('matching_recurrence', tables.get('recurrence', pd.DataFrame()))
display(recurrence)


## Complete primary and secondary rules

In [ ]:
rules = tables.get('rules', pd.DataFrame())
rule_columns = [column for column in ['reference_year', 'patient_split_seed', 'factor_family_uid', 'member_sae_seed', 'rule_source', 'activation_target', 'compatibility_H', 'cutoff', 'rule_text', 'precision', 'recall', 'f2', 'lift', 'target_prevalence', 'prediction_prevalence', 'cohort_size', 'geometric_factor_recurrence', 'semantic_family_recurrence', 'target_role', 'valid', 'failure_reason'] if column in rules]
display(rules[rule_columns] if rule_columns else rules)


## Anchor matrices and artifact-backed sensitivity selectors

In [ ]:
def select_view(frame, *, cohort='all_comer', matching_view='cosine_qualified', cosine=HEADLINE_COSINE, overlap=HEADLINE_OVERLAP, rule_source=None):
    selected = frame.copy()
    filters = {'cohort_view': cohort, 'matching_view': matching_view, 'cosine_threshold': cosine, 'overlap_threshold': overlap}
    if rule_source is not None:
        filters['rule_source'] = rule_source
    for column, value in filters.items():
        if column in selected and value is not None:
            selected = selected[selected[column].eq(value)]
    return selected

def plot_triangular(frame, metric, title):
    if frame.empty or not {'reference_year', 'test_year', metric}.issubset(frame):
        print(f'Unavailable: {metric}')
        return
    matrix = frame.pivot_table(index='reference_year', columns='test_year', values=metric, aggfunc='mean')
    support_matrix = frame.pivot_table(index='reference_year', columns='test_year', values=metric, aggfunc='count')
    display(support_matrix.style.set_caption(f'{title} support'))
    fig, axis = plt.subplots(figsize=(9, 5))
    image = axis.imshow(matrix.to_numpy(float), aspect='auto')
    axis.set(title=title, xlabel='Test year', ylabel='Reference year', xticks=range(len(matrix.columns)), yticks=range(len(matrix.index)), xticklabels=matrix.columns, yticklabels=matrix.index)
    fig.colorbar(image, ax=axis)
    plt.show()

anchor_performance = select_view(support)
for metric in ('macro_f1', 'death_f1'):
    plot_triangular(anchor_performance, metric, f'Anchor all-comer {metric}')


## Temporal-distance summaries, uncertainty, lead-lag, and exploratory change points

In [ ]:
for name in ('distance_summaries', 'uncertainty', 'lead_lag', 'change_points'):
    frame = tables.get(name, pd.DataFrame())
    if not frame.empty and 'selection_scope' in frame:
        post_hoc = frame['selection_scope'].eq('post_hoc_future')
        if post_hoc.any():
            print(f'{name}: {int(post_hoc.sum())} post-hoc future sensitivity rows')
    display(frame)
trajectories = tables.get('factor_year_metrics', pd.DataFrame())
if not trajectories.empty and 'factor_count' in trajectories:
    display(trajectories[trajectories['factor_count'] < 10].assign(exploratory=True))
